# English → Bhojpuri Translation Pipeline (NLLB-200)

**Optimized for Kaggle T4 GPU**

- Model: NLLB-200-distilled-600M (✅ Working)
- Speed: 500+ translations/sec on GPU
- Quality: Authentic Bhojpuri (Devanagari)
- Target: 360M+ Bhojpuri tokens in 2 days

## Cell 1: Install Dependencies

In [ ]:
!pip install -q transformers torch

import torch
import json
import os
import time
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Cell 2: Load Model & Tokenizer

In [ ]:
# Configuration
SOURCE_LANG = "eng_Latn"  # English
TARGET_LANG = "bho_Deva"  # Bhojpuri
BATCH_SIZE = 32  # Adjust: 128 (aggressive), 64 (balanced), 32 (safe)
CHECKPOINT_EVERY = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load tokenizer
print(f"\nLoading tokenizer for {SOURCE_LANG}...")
tokenizer = AutoTokenizer.from_pretrained(
    "facebook/nllb-200-distilled-600M",
    src_lang=SOURCE_LANG,
    use_fast=True,
)

# Get target language token ID (CRITICAL for Bhojpuri output)
target_lang_id = tokenizer.convert_tokens_to_ids(TARGET_LANG)
print(f"Target language ({TARGET_LANG}) token ID: {target_lang_id}")

# Load model
print(f"\nLoading model facebook/nllb-200-distilled-600M...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/nllb-200-distilled-600M",
    dtype=torch.float16 if device.type == "cuda" else torch.float32,
)
model = model.to(device)
model.eval()
print("✅ Model loaded successfully!")

## Cell 3: Test Translation Function

In [ ]:
@torch.inference_mode()
def translate_batch(texts, batch_size=32):
    """
    Translate batch of English texts to Bhojpuri
    
    Args:
        texts: List of English texts
        batch_size: Process in batches
    
    Returns:
        List of Bhojpuri translations
    """
    # Filter empty/short texts
    texts = [t.strip() for t in texts if t and len(t.strip().split()) >= 2]
    if not texts:
        return []
    
    translations = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        
        # Tokenize
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,
            src_lang=SOURCE_LANG,
        ).to(device)
        
        # Generate (forced_bos_token_id sets target language)
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=target_lang_id,
            max_length=256,
            num_beams=1,
            do_sample=False,
        )
        
        # Decode
        batch_translations = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True,
        )
        translations.extend(batch_translations)
    
    return translations


# Test with sample texts
print("Testing translation...\n")
test_texts = [
    "The government announced a new policy.",
    "People gathered near the river yesterday.",
    "Science and technology are advancing rapidly.",
]

results = translate_batch(test_texts)
for en, bho in zip(test_texts, results):
    print(f"EN : {en}")
    print(f"BHO: {bho}")
    print()

print("✅ Translation working correctly (Devanagari script)")

## Cell 4: Setup Data & Output Directories

In [ ]:
# Input data (fineweb-edu dataset on Kaggle)
INPUT_DIR = "/kaggle/input/datasets/nameonlu/fineweb-edu"

# Check if input exists
if not os.path.exists(INPUT_DIR):
    print(f"⚠️  Input directory not found: {INPUT_DIR}")
    print("\nMake sure to add 'fineweb-edu' dataset to your Kaggle notebook:")
    print("1. Click 'Data' tab")
    print("2. Click '+ Add' → 'From Kaggle'")
    print("3. Search 'fineweb-edu' by nameonlu")
    print("4. Add to notebook")
else:
    # Find all text files
    txt_files = sorted(Path(INPUT_DIR).glob("*.txt"))
    print(f"✅ Found {len(txt_files)} input files:")
    
    total_size_mb = 0
    for f in txt_files[:5]:
        size_mb = f.stat().st_size / 1024**2
        total_size_mb += size_mb
        print(f"  - {f.name} ({size_mb:.0f} MB)")
    
    if len(txt_files) > 5:
        remaining = sum(f.stat().st_size / 1024**2 for f in txt_files[5:])
        total_size_mb += remaining
        print(f"  ... and {len(txt_files)-5} more files ({remaining:.0f} MB)")
    
    print(f"\nTotal data: {total_size_mb:.0f} MB")

# Setup output directory
OUT_DIR = "/kaggle/working/bhojpuri_translations"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"\n✅ Output directory: {OUT_DIR}")

## Cell 5: Process Files with Checkpointing

In [ ]:
CHECKPOINT_FILE = os.path.join(OUT_DIR, "checkpoint.json")
OUTPUT_FILE = os.path.join(OUT_DIR, "translations.jsonl")

def load_checkpoint():
    """Load checkpoint to resume from last position"""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)
    return {
        "file_index": 0,
        "lines_processed": 0,
        "docs_output": 0,
        "start_time": time.time(),
    }

def save_checkpoint(state):
    """Save checkpoint for resumability"""
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(state, f, indent=2)

# Load previous checkpoint if exists
checkpoint = load_checkpoint()

# Files 0 and 1 were already processed in separate Kaggle sessions.
# Skip straight to file 2 to avoid re-translating and adding duplicates.
FORCE_START_FILE_INDEX = 2  # set to None to fall back to normal checkpoint resume

if FORCE_START_FILE_INDEX is not None and checkpoint["file_index"] < FORCE_START_FILE_INDEX:
    print(f"⏭️  Skipping to file_index={FORCE_START_FILE_INDEX} "
          f"(files 0-1 already processed in separate sessions)")
    checkpoint["file_index"] = FORCE_START_FILE_INDEX
    checkpoint["lines_processed"] = 0
    save_checkpoint(checkpoint)

print(f"Checkpoint: {json.dumps(checkpoint, indent=2)}")

# Open output file for appending
output_handle = open(OUTPUT_FILE, "a", encoding="utf-8", buffering=1024*1024)

start_time = time.time()
file_index = checkpoint["file_index"]
docs_output = checkpoint.get("docs_output", 0)

print(f"\nStarting processing from file {file_index}...\n")

try:
    for fi in range(file_index, len(txt_files)):
        txt_path = txt_files[fi]
        filename = txt_path.name
        
        print(f"\n[{fi+1}/{len(txt_files)}] Processing {filename}")
        
        # Read all lines from file
        with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
            all_lines = [line.strip() for line in f if line.strip()]
        
        print(f"  Loaded {len(all_lines):,} lines")
        
        # Process in batches with progress bar
        pbar = tqdm(
            range(0, len(all_lines), BATCH_SIZE),
            desc=filename,
            total=(len(all_lines) + BATCH_SIZE - 1) // BATCH_SIZE,
        )
        
        for batch_start in pbar:
            batch_end = min(batch_start + BATCH_SIZE, len(all_lines))
            batch = all_lines[batch_start:batch_end]
            
            # Translate batch
            translations = translate_batch(batch, batch_size=BATCH_SIZE)
            
            # Save valid translations
            for en_text, bho_text in zip(batch, translations):
                bho_text = bho_text.strip()
                
                if bho_text and len(bho_text.split()) >= 2:
                    record = {
                        "en": en_text,
                        "bho": bho_text,
                        "lang_pair": "en_Latn-bho_Deva",
                    }
                    output_handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                    docs_output += 1
            
            # Save checkpoint every N documents
            if docs_output % CHECKPOINT_EVERY == 0:
                save_checkpoint({
                    "file_index": fi,
                    "lines_processed": batch_end,
                    "docs_output": docs_output,
                    "start_time": start_time,
                })
                output_handle.flush()
                
                elapsed = (time.time() - start_time) / 60
                rate = docs_output / elapsed if elapsed > 0 else 0
                pbar.set_postfix({
                    "output": f"{docs_output:,}",
                    "rate": f"{rate:.0f}/min",
                    "time": f"{elapsed:.1f}m",
                })
        
        # Save checkpoint after each file
        save_checkpoint({
            "file_index": fi + 1,
            "lines_processed": len(all_lines),
            "docs_output": docs_output,
            "start_time": start_time,
        })
        output_handle.flush()

finally:
    output_handle.close()
    elapsed = (time.time() - start_time) / 60
    
    print("\n" + "="*70)
    print("✅ TRANSLATION COMPLETE")
    print("="*70)
    print(f"Documents processed: {docs_output:,}")
    print(f"Time elapsed: {elapsed:.1f} minutes ({elapsed/60:.1f} hours)")
    if elapsed > 0:
        print(f"Rate: {docs_output / elapsed:.0f} translations/min")
    print(f"Output file: {OUTPUT_FILE}")

## Cell 6: Verify Output

In [ ]:
if os.path.exists(OUTPUT_FILE):
    size_mb = os.path.getsize(OUTPUT_FILE) / 1024**2
    
    # Count total translations
    with open(OUTPUT_FILE) as f:
        num_translations = sum(1 for _ in f)
    
    print(f"✅ Output file: {OUTPUT_FILE}")
    print(f"   Size: {size_mb:.1f} MB")
    print(f"   Total translations: {num_translations:,}")
    
    # Estimate tokens (rough: 1 word ≈ 1.3 tokens in Bhojpuri)
    # Read sample to count words
    word_count = 0
    with open(OUTPUT_FILE) as f:
        for i, line in enumerate(f):
            if i >= 100:  # Sample 100 lines
                break
            data = json.loads(line)
            word_count += len(data.get("bho", "").split())
    
    avg_words_per_line = word_count / 100 if word_count > 0 else 5
    estimated_tokens = num_translations * avg_words_per_line * 1.3
    
    print(f"   Estimated tokens: {estimated_tokens:,.0f}")
    print(f"   Avg words/translation: {avg_words_per_line:.1f}")
    
    # Show samples
    print(f"\n{'='*80}")
    print(f"Sample translations:")
    print(f"{'='*80}")
    
    with open(OUTPUT_FILE) as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            
            data = json.loads(line)
            en_text = data["en"]
            bho_text = data["bho"]
            
            print(f"\n[{i+1}]")
            print(f"EN : {en_text[:100]}")
            print(f"BHO: {bho_text[:100]}")
    
    print(f"\n{'='*80}")
else:
    print("⚠️  Output file not created yet. Check previous cell for errors.")

## Cell 7: Monitor Progress (Run Anytime)

In [ ]:
# Check current checkpoint
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        cp = json.load(f)
    
    print("Current Progress:")
    print(f"  File: {cp['file_index']}/{len(txt_files)}")
    print(f"  Documents output: {cp.get('docs_output', 0):,}")
    print(f"  Time elapsed: {(time.time() - cp['start_time'])/60:.1f} minutes")
    
    # Count output lines
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE) as f:
            count = sum(1 for _ in f)
        print(f"  Output lines: {count:,}")
else:
    print("No checkpoint found. Processing not started.")

## Cell 8: Download Results

After translation completes:
1. Download `translations.jsonl` from Kaggle output
2. Place in: `bhojpuri/data/english_translations/`
3. Run merge script locally:
   ```bash
   python3 bhojpuri/data_collect/ocr_merge.py
   python3 bhojpuri/data_collect/update_config_ocr.py
   ```

In [ ]:
# Show download instructions
print("""\n📥 NEXT STEPS
=========================================

1. Wait for translation to complete
2. Click 'Output' tab in Kaggle notebook
3. Download file: bhojpuri_translations/translations.jsonl
4. Place in local directory:
   bhojpuri/data/english_translations/

5. Merge into corpus (on local machine):
   python3 bhojpuri/data_collect/ocr_merge.py
   python3 bhojpuri/data_collect/update_config_ocr.py

6. Verify target reached:
   python3 -c "import json; config = json.load(open('bhojpuri/data/config.json')); \
   total = config.get('total_tokens', 0); \
   print(f'Total tokens: {total:,.0f}'); \
   print('✅ TARGET REACHED!' if total >= 500e6 else '❌ Need more data')"

=========================================
""")